# Text-to-Brain Metrics v2 (Colab)

## Colab Setup

Run this setup cell once in Colab before the rest of the notebook. It installs the pushed branch that contains the Colab loaders and evaluation code.

In [ ]:
import importlib
import sys
import subprocess

BRANCH = "new_text-to-brain_metrics"
REPO_URL = f"git+https://github.com/neurovlm/neurovlm.git@{BRANCH}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", REPO_URL])

SPIN_PACKAGES = [
    "accelerate",
    "nibabel",
    "nilearn",
    "templateflow",
    "neuromaps",
    "brainspace",
    "nitransforms",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-cache-dir",
    *SPIN_PACKAGES,
])
importlib.invalidate_caches()

# If any Hugging Face files/models are private, uncomment these two lines:
# from huggingface_hub import notebook_login
# notebook_login()


## Spin-Test Dependency Check

Run this after setup and before the imports/config cells. If this fails, restart the runtime after the setup cell and run it again.

In [ ]:
import importlib
import importlib.metadata as importlib_metadata
import sys
import traceback

SPIN_MODULES = ["neuromaps", "brainspace", "nitransforms", "nibabel", "nilearn", "templateflow"]

try:
    for package in SPIN_MODULES:
        module = importlib.import_module(package)
        version = importlib_metadata.version(package)
        print(f"{package}: {version} ({getattr(module, '__file__', 'built-in')})")

    from neuromaps import transforms
    from brainspace.null_models import SpinPermutations

    assert hasattr(transforms, "mni152_to_fsaverage"), "neuromaps.transforms.mni152_to_fsaverage is unavailable"
    print("Spin-test dependencies OK")
    print("Python:", sys.executable)
except Exception:
    print("Spin-test dependency check failed. Full traceback follows:")
    traceback.print_exc()
    raise


In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from scipy.stats import spearmanr, rankdata

from neurovlm import NeuroVLM
from neurovlm.data import load_dataset, load_latent, load_masker
from neurovlm.metrics import (
    pearson_correlation,
    dice_percentile,
    nct_dice_spin_test_surface,
    mni152_to_fsaverage_arrays,
    precompute_spin_permutations,
)


In [ ]:
MAX_T2B = None        # full available networks, PubMed test, and NeuroVault sets
RUN_NETWORKS = True
RUN_PUBMED = True
RUN_NEUROVAULT = True

DICE_PCT = 90
DICE_SENSITIVITY_PCTS = [80, 85, 90, 95]
SPIN_USE_NEUROMAPS = True   # Colab run: compute surface spin-test p-values
SPIN_REQUIRE_NEUROMAPS = True  # fail loudly instead of silently writing blank spin p-values
SPIN_TEST_N_PERM = 1000
SPIN_TEST_RANDOM_STATE = 13
SPIN_FSAVERAGE_DENSITY = "41k"
SPIN_TRANSFORM_METHOD = "linear"


PUBMED_SURFACE_FILTER_ENABLED = True
PUBMED_MIN_CORTICAL_TOP_MASS_FRACTION = 0.50
NEUROVAULT_IMAGE_SELECTION = "argmax_pearson"  # "first" or "argmax_pearson"
T2B_RANDOM_BASELINE_N = 25
T2B_RANDOM_BASELINE_SEED = 13
T2B_RANDOM_BASELINE_SPEARMAN_MAX_VOXELS = 10000

OUTPUT_DIR = Path("docs/03_evaluation/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
T2B_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

nvlm = NeuroVLM(device=T2B_DEVICE)
masker = load_masker()
print(f"NeuroVLM inference device: {nvlm.device}")


if SPIN_USE_NEUROMAPS:
    precompute_spin_permutations(
        density=SPIN_FSAVERAGE_DENSITY,
        n_perm=SPIN_TEST_N_PERM,
        random_state=SPIN_TEST_RANDOM_STATE,
    )
    print(f"Precomputed {SPIN_TEST_N_PERM} fsaverage {SPIN_FSAVERAGE_DENSITY} spin permutations.")

print("Ready.")


## Evaluation Setup

Text-to-brain predicts a brain map from reference text and compares it with a target brain map. For PubMed, surface Dice and spin-test p-values are only meaningful when the target activation is substantially cortical, because the spin workflow projects MNI volume maps to fsaverage surface. PubMed targets with mostly subcortical top activation are therefore kept for whole-brain Pearson/Spearman but excluded from surface Dice/spin summaries.

For NeuroVault, a paper can have multiple linked images. `NEUROVAULT_IMAGE_SELECTION = "argmax_pearson"` generates one predicted map from the paper text and selects the linked NeuroVault image with the highest Pearson correlation to that prediction. The selected image index and number of candidates are saved for audit. This is a paper-level evaluation instead of arbitrarily using the first uploaded image.


In [ ]:
from neurovlm.evaluation_notebook_utils import load_network_label_resources

network_labels_df, network_info, NETWORK_TEST_SET_SOURCE = load_network_label_resources()

DISPLAY_TO_KEY = dict(zip(network_info["display"], network_info["network_key"]))
KEY_TO_DISPLAY = dict(zip(network_info["network_key"], network_info["display"]))
SHORT_LABELS = dict(zip(network_info["display"], network_info["short_definition"]))
LONG_LABELS = dict(zip(network_info["display"], network_info["long_definition"]))

print(f"Loaded {len(network_info)} canonical network labels from {NETWORK_TEST_SET_SOURCE}")
display(network_info[["network_key", "display", "short_definition"]])


In [ ]:
from neurovlm.evaluation_notebook_utils import build_labeled_network_data

networks_data, all_net_latents = build_labeled_network_data(network_labels_df)

print(f"Networks loaded: {len(networks_data)} labeled maps across {len(all_net_latents)} atlases")
display(pd.DataFrame([
    {"sample": name, "network_key": d["network_key"], "display": d["display"]}
    for name, d in networks_data.items()
]).head())


In [ ]:
from neurovlm.evaluation_notebook_utils import build_pubmed_t2b_eval

pubmed_eval = build_pubmed_t2b_eval(MAX_T2B)
print(f"PubMed samples: {len(pubmed_eval)}")


In [ ]:
from neurovlm.evaluation_notebook_utils import build_neurovault_t2b_eval

neurovault_eval = build_neurovault_t2b_eval(MAX_T2B)
print(f"NeuroVault samples: {len(neurovault_eval)}")
print(f"NeuroVault image selection mode: {NEUROVAULT_IMAGE_SELECTION}")
if len(neurovault_eval):
    n_multi = sum(d["n_candidate_images"] > 1 for d in neurovault_eval)
    print(f"NeuroVault papers with multiple candidate images in eval set: {n_multi:,}")


In [ ]:
from neurovlm.evaluation_notebook_utils import (
    build_surface_eligibility_mask,
    evaluate_t2b_sample,
)

SURFACE_ELIGIBILITY_MASK, SURFACE_ELIGIBILITY_METHOD = build_surface_eligibility_mask(masker)
print(f"Surface eligibility mask: {SURFACE_ELIGIBILITY_METHOD}; cortical voxels={int(SURFACE_ELIGIBILITY_MASK.sum()):,}")


def run_t2b(
    name,
    text_input,
    true_latent,
    dataset_name="unknown",
    candidate_latents=None,
    candidate_indices=None,
    neurovault_selection_mode="first",
):
    return evaluate_t2b_sample(
        nvlm=nvlm,
        masker=masker,
        name=name,
        text_input=text_input,
        true_latent=true_latent,
        dataset_name=dataset_name,
        candidate_latents=candidate_latents,
        candidate_indices=candidate_indices,
        neurovault_selection_mode=neurovault_selection_mode,
        dice_pct=DICE_PCT,
        surface_mask=SURFACE_ELIGIBILITY_MASK,
        surface_eligibility_method=SURFACE_ELIGIBILITY_METHOD,
        pubmed_surface_filter_enabled=PUBMED_SURFACE_FILTER_ENABLED,
        pubmed_min_cortical_top_mass_fraction=PUBMED_MIN_CORTICAL_TOP_MASS_FRACTION,
        spin_use_neuromaps=SPIN_USE_NEUROMAPS,
        spin_require_neuromaps=SPIN_REQUIRE_NEUROMAPS,
        spin_test_n_perm=SPIN_TEST_N_PERM,
        spin_test_random_state=SPIN_TEST_RANDOM_STATE,
        spin_fsaverage_density=SPIN_FSAVERAGE_DENSITY,
        spin_transform_method=SPIN_TRANSFORM_METHOD,
    )


In [ ]:
t2b_frames = []

if RUN_NETWORKS:
    records = []
    for net_name, d in tqdm(networks_data.items(), desc="Networks T2B"):
        rec = run_t2b(net_name, d["long_gt"], d["latent"], dataset_name="networks")
        if rec is not None:
            records.append(rec)
    t2b_net_df = pd.DataFrame(records)
    t2b_net_df["dataset"] = "networks"
    t2b_frames.append(t2b_net_df)

if RUN_PUBMED:
    records = []
    for d in tqdm(pubmed_eval, desc="PubMed T2B"):
        rec = run_t2b(str(d["pmid"]), d["short_gt"] + " [SEP] " + d["long_gt"], d["latent"], dataset_name="pubmed")
        if rec is not None:
            records.append(rec)
    t2b_pubmed_df = pd.DataFrame(records)
    t2b_pubmed_df["dataset"] = "pubmed"
    t2b_frames.append(t2b_pubmed_df)

if RUN_NEUROVAULT:
    records = []
    for d in tqdm(neurovault_eval, desc="NeuroVault T2B"):
        rec = run_t2b(
            str(d["doi"]),
            d["short_gt"] + " [SEP] " + d["long_gt"],
            d["latent"],
            dataset_name="neurovault",
            candidate_latents=d.get("candidate_latents"),
            candidate_indices=d.get("candidate_image_indices"),
            neurovault_selection_mode=NEUROVAULT_IMAGE_SELECTION,
        )
        if rec is not None:
            records.append(rec)
    t2b_nv_df = pd.DataFrame(records)
    t2b_nv_df["dataset"] = "neurovault"
    t2b_frames.append(t2b_nv_df)

t2b_all = pd.concat(t2b_frames, ignore_index=True)
t2b_all.head()


In [ ]:
from neurovlm.evaluation_notebook_utils import add_random_correlation_baseline

t2b_all = add_random_correlation_baseline(
    t2b_all,
    n_random=T2B_RANDOM_BASELINE_N,
    seed=T2B_RANDOM_BASELINE_SEED,
    max_voxels=T2B_RANDOM_BASELINE_SPEARMAN_MAX_VOXELS,
)
_public_cols = [c for c in t2b_all.columns if not c.startswith("_")]
t2b_all[_public_cols].to_csv(OUTPUT_DIR / "text_to_brain_metrics_v2.csv", index=False)
t2b_all[_public_cols].head()


## Metric Summary

- **Pearson r** checks linear whole-brain voxelwise similarity between predicted and target maps. The random baseline uses a fixed voxel subset for speed; `pearson_minus_random > 0` means the prediction is more similar to its own target than to random targets from the same dataset.
- **Spearman rho** checks rank-order whole-brain similarity. The random baseline also uses the fixed voxel subset; `spearman_minus_random > 0` is the main interpretation.
- **Dice pct** checks overlap among the strongest activation regions at percentile `DICE_PCT`. PubMed samples that are mostly subcortical are excluded because surface projection would not fairly represent them.
- **Spin p-value** tests whether surface Dice is stronger than spatially rotated null maps. Lower is better; `p < 0.05` is conventionally significant. It is only interpreted for surface-eligible samples.


In [ ]:
dice_col = f"dice_pct{DICE_PCT}"
metric_cols = [
    "pearson_r", "pearson_baseline_actual", "pearson_random_mean", "pearson_minus_random", "pearson_random_percentile",
    "spearman_rho", "spearman_baseline_actual", "spearman_random_mean", "spearman_minus_random", "spearman_random_percentile",
    dice_col, "spin_p_value",
]
summary = t2b_all.groupby("dataset")[metric_cols].agg(["mean", "std", "count"]).round(3)
display(summary)
display(t2b_all.groupby("dataset")["spin_significant"].sum().rename("n_spin_p_lt_0_05"))
display(t2b_all.groupby(["dataset", "surface_metric_eligible", "surface_metric_skip_reason"]).size().rename("n"))
display(t2b_all.groupby(["dataset", "spin_method"]).size().rename("n"))
if "neurovault" in set(t2b_all["dataset"]):
    display(t2b_all[t2b_all["dataset"] == "neurovault"][["n_candidate_images", "selected_candidate_position", "selected_candidate_pearson"]].describe().round(3))


## Correlation Random Baseline Plot

This plot compares each sample's actual Pearson/Spearman score against random target maps from the same dataset, using a fixed voxel subset for speed. If actual scores are above random, the model is capturing sample-specific brain structure rather than only generic dataset-level activation patterns.


In [ ]:
from neurovlm.evaluation_notebook_utils import finite_box_values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric, random_col, title in [
    (axes[0], "pearson_baseline_actual", "pearson_random_mean", "Pearson r (baseline voxel subset)"),
    (axes[1], "spearman_baseline_actual", "spearman_random_mean", "Spearman rho (baseline voxel subset)"),
]:
    data = []
    labels = []
    positions = []
    pos = 1
    for dataset, sub in t2b_all.groupby("dataset"):
        data.extend([finite_box_values(sub[metric]), finite_box_values(sub[random_col])])
        labels.extend([f"{dataset} actual", f"{dataset} random"])
        positions.extend([pos, pos + 0.8])
        pos += 2.2
    ax.boxplot(data, positions=positions, labels=labels, showmeans=True)
    ax.axhline(0, color="0.4", linestyle="--", linewidth=1)
    ax.set_title(f"{title}: actual vs random target")
    ax.set_ylabel(title)
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "t2b_correlation_random_baseline.png", dpi=150, bbox_inches="tight")
plt.show()


## Dice Threshold Sensitivity

This recomputes Dice and spin significance across several percentile thresholds. Stable conclusions across thresholds are stronger than a result that only appears at one arbitrary cutoff. PubMed rows failing the cortical surface-eligibility check remain excluded at every threshold.


In [ ]:
from neurovlm.evaluation_notebook_utils import sensitivity_rows_for_df

sensitivity_records = []
for dataset, sub in t2b_all.groupby("dataset"):
    sensitivity_records.extend(
        sensitivity_rows_for_df(
            sub.reset_index(drop=True),
            dataset,
            pcts=DICE_SENSITIVITY_PCTS,
            spin_test_n_perm=SPIN_TEST_N_PERM,
            spin_test_random_state=SPIN_TEST_RANDOM_STATE,
            spin_fsaverage_density=SPIN_FSAVERAGE_DENSITY,
            spin_require_neuromaps=SPIN_REQUIRE_NEUROMAPS,
        )
    )
t2b_sensitivity_df = pd.DataFrame(sensitivity_records)
display(t2b_sensitivity_df.groupby(["dataset", "pct"]).agg(dice_mean=("dice", "mean"), dice_std=("dice", "std"), sig_rate=("spin_significant", "mean"), n_surface_metric=("dice", "count"), n_total=("sample", "count")).round(3))


## Metric Distribution Plot

Pearson and Spearman are shown as improvement over the random baseline. Values above zero indicate better-than-random target matching. Dice and spin p-value use only surface-eligible rows; missing PubMed values usually indicate mostly subcortical target activation.


In [ ]:
from neurovlm.evaluation_notebook_utils import finite_box_values

dice_col = f"dice_pct{DICE_PCT}"
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
plot_df = t2b_all.copy()
for ax, metric, title in zip(axes, ["pearson_minus_random", "spearman_minus_random", dice_col, "spin_p_value"], ["Pearson minus random", "Spearman minus random", f"Dice pct {DICE_PCT}", "Spin p-value"]):
    groups = [finite_box_values(g[metric]) for _, g in plot_df.groupby("dataset")]
    labels = [k for k, _ in plot_df.groupby("dataset")]
    ax.boxplot(groups, labels=labels, showmeans=True)
    if metric in {"pearson_minus_random", "spearman_minus_random"}:
        ax.axhline(0, color="0.4", linestyle="--", linewidth=1, label="random baseline")
        ax.legend(frameon=False)
    if metric == "spin_p_value":
        ax.axhline(0.05, color="crimson", linestyle="--", linewidth=1, label="p = 0.05")
        ax.legend(frameon=False)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "t2b_metric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


## Dice Threshold Sensitivity Plot

The left panel shows mean Dice as the activation percentile threshold changes; the right panel shows the fraction of samples with spin-test `p < 0.05`. If the conclusion changes sharply across thresholds, the Dice result is threshold-sensitive. Surface-ineligible PubMed rows remain excluded from Dice/spin at all thresholds.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary_plot = t2b_sensitivity_df.groupby(["dataset", "pct"]).agg(
    dice_mean=("dice", "mean"),
    dice_sem=("dice", lambda x: x.std() / np.sqrt(max(x.count(), 1))),
    sig_rate=("spin_significant", "mean"),
).reset_index()
for dataset, sub in summary_plot.groupby("dataset"):
    axes[0].errorbar(sub["pct"], sub["dice_mean"], yerr=sub["dice_sem"], marker="o", label=dataset)
    axes[1].plot(sub["pct"], sub["sig_rate"], marker="o", label=dataset)
axes[0].set_xlabel("Percentile threshold")
axes[0].set_ylabel("Mean Dice")
axes[0].set_title("Dice threshold sensitivity")
axes[1].set_xlabel("Percentile threshold")
axes[1].set_ylabel("Fraction p < 0.05")
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title("Spin-test significance rate")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "t2b_dice_pvalue_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()


## Dice vs Spin Significance

This scatterplot shows whether larger activation overlap also corresponds to stronger spatial significance. The horizontal line marks `p = 0.05`; points above it are significant after the spin-test null model. Color shows Pearson improvement over random target maps.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_df = t2b_all.dropna(subset=[dice_col, "spin_p_value", "pearson_minus_random"])
if len(plot_df):
    scatter = ax.scatter(plot_df[dice_col], -np.log10(plot_df["spin_p_value"].clip(lower=1e-6)), c=plot_df["pearson_minus_random"], cmap="viridis", s=55, alpha=0.8)
    cb = fig.colorbar(scatter, ax=ax)
    cb.set_label("Pearson minus random")
else:
    ax.text(0.5, 0.5, "No eligible finite Dice/spin rows", ha="center", va="center", transform=ax.transAxes)
ax.axhline(-np.log10(0.05), color="crimson", linestyle="--", linewidth=1)
ax.set_xlabel(f"Dice pct {DICE_PCT}")
ax.set_ylabel("-log10(spin p-value)")
ax.set_title("Dice overlap and spatial significance")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "t2b_dice_vs_pvalue.png", dpi=150, bbox_inches="tight")
plt.show()
